In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType
import re

In [0]:
%run /Workspace/Users/yanquiel@softserve.academy/ecommerce-bronze-platform-/notebooks/utilities

In [0]:
dbutils.widgets.text("catalog", "dbr_dev", "Catalog")
dbutils.widgets.text("eventhub_name", "evh_brazilian_ecommerce", "Event Hub Name")

catalog = dbutils.widgets.get("catalog")
eventhub_name = dbutils.widgets.get("eventhub_name")

In [0]:
connection_string = dbutils.secrets.get(
    scope="ecommerce-bronze-scope",
    key="evh-brazilian-ecommerce"
)


In [0]:
namespace_match = re.search(r"sb://([^./]+)\.servicebus\.windows\.net", connection_string)
namespace = namespace_match.group(1)
bootstrap_servers = f"{namespace}.servicebus.windows.net:9093"

# NOTE: must use the shaded class name (kafkashaded.org.apache.kafka...) here.
# Databricks Runtime relocates the Kafka client classes internally, so the unshaded
# class name (org.apache.kafka...) throws a ClassNotFoundException when the
# AdminClient tries to instantiate the login module, surfacing as the generic
# "Failed to create new KafkaAdminClient" error.
sasl_config = (
    'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
    f'username="$ConnectionString" password="{connection_string}";'
)

kafka_options = {
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": sasl_config,
    "subscribe": eventhub_name,
    "startingOffsets": "latest",
    "failOnDataLoss": "false",
}

In [0]:
raw_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", bootstrap_servers)
    .option("kafka.security.protocol", "SASL_SSL")
    .options(**kafka_options)
    .load()
)

In [0]:
order_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("price", DoubleType(), True),
    StructField("order_timestamp", TimestampType(), True),
    StructField("discount_code", StringType(), True),   # not sent yet will start arriving null until the producer adds it
])

In [0]:
parsed_df = (
    raw_df
    .select(
        F.col("value").cast("string").alias("json_payload"),
        F.col("partition").alias("kafka_partition"),
        F.col("offset").alias("kafka_offset"),
    )
    .withColumn("data", F.from_json(F.col("json_payload"), order_schema))
    .select("data.*", "kafka_partition", "kafka_offset")
    .withColumn("source", F.lit(eventhub_name))
    .withColumn("ingestion_timestamp", F.current_timestamp())
)

In [0]:
checkpoint_path = f"/Volumes/{catalog}/{bronze_schema}/landing/checkpoints/brz_orders"

query = (
    parsed_df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .trigger(processingTime="10 seconds")
    .toTable(f"{catalog}.{bronze_schema}.brz_orders")
)

In [0]:
%sql
SELECT * FROM dbr_dev.brazilian_ecommerce_bronze.brz_orders ORDER BY ingestion_timestamp DESC LIMIT 20;

In [0]:
query.status

In [0]:

query.lastProgress